In [ ]:
# Complete Quantum Circuit Optimization Demo
# All functionality in one cell

# Import necessary modules
import sys
sys.path.append('..')

from modul.circuit import Circuit
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# 1. CREATE INITIAL CIRCUIT WITH PP-H-PP-H-PP PATTERN
print("1. INITIAL CIRCUIT WITH PP-H-PP-H-PP PATTERN")
print("=" * 60)

circuit = Circuit(total_qubits=3)
qc = circuit.get_circuit()
phases = np.random.rand(3, 6) * 2 * np.pi

for qubit in range(3):
    # PP
    qc.p(phases[qubit, 0], qubit)
    qc.p(phases[qubit, 1], qubit)
    # H
    qc.h(qubit)
    # PP
    qc.p(phases[qubit, 2], qubit)
    qc.p(phases[qubit, 3], qubit)
    # H
    qc.h(qubit)
    # PP
    qc.p(phases[qubit, 4], qubit)
    qc.p(phases[qubit, 5], qubit)

print(qc.draw(output='text'))

# 2. CREATE INPUT AND TRAINING MATRICES
print("\n\n2. INPUT AND TRAINING MATRICES")
print("=" * 60)

input_matrix = np.eye(3)
print("Input Matrix (Identity):")
print(input_matrix)

training_matrix = np.random.rand(3, 3)
print("\nTraining Matrix (Random):")
print(training_matrix)

# 3. CREATE CIRCUIT WITH MATRIX VALUES (IP-TP-H-IP-TP-H-IP-TP)
print("\n\n3. CIRCUIT WITH IP-TP-H-IP-TP-H-IP-TP PATTERN")
print("=" * 60)

matrix_circuit = Circuit(total_qubits=3)
qc_matrix = matrix_circuit.get_circuit()

for qubit in range(3):
    # IP1-TP1
    qc_matrix.p(input_matrix[qubit, 0], qubit)
    qc_matrix.p(training_matrix[qubit, 0], qubit)
    # H
    qc_matrix.h(qubit)
    # IP2-TP2
    qc_matrix.p(input_matrix[qubit, 1], qubit)
    qc_matrix.p(training_matrix[qubit, 1], qubit)
    # H
    qc_matrix.h(qubit)
    # IP3-TP3
    qc_matrix.p(input_matrix[qubit, 2], qubit)
    qc_matrix.p(training_matrix[qubit, 2], qubit)

print(qc_matrix.draw(output='text'))

# 4. INITIAL MEASUREMENT AND PROBABILITY DISTRIBUTION
print("\n\n4. INITIAL MEASUREMENT")
print("=" * 60)

qc_matrix.measure_all()
simulator = AerSimulator()
compiled_circuit = transpile(qc_matrix, simulator)
job = simulator.run(compiled_circuit, shots=1000)
result = job.result()
counts = result.get_counts(compiled_circuit)

print("Initial measurement results:")
for state, count in sorted(counts.items()):
    print(f"|{state}>: {count/1000:.2%}")

# 5. DEFINE TARGET STATE
target_state = max(counts, key=counts.get)
target_prob_initial = counts[target_state] / 1000
target_index = int(target_state.replace(' ', ''), 2)

print(f"\nTarget State: |{target_state}> with initial probability {target_prob_initial:.2%}")

# 6. OPTIMIZATION WITH SCIPY
print("\n\n5. OPTIMIZATION PROCESS")
print("=" * 60)

loss_history = []
prob_history = []

def create_circuit_with_phases(input_mat, training_phases_flat):
    training_phases_mat = training_phases_flat.reshape(3, 3)
    qc = QuantumCircuit(3)
    
    for qubit in range(3):
        qc.p(float(input_mat[qubit, 0]), qubit)
        qc.p(float(training_phases_mat[qubit, 0]), qubit)
        qc.h(qubit)
        qc.p(float(input_mat[qubit, 1]), qubit)
        qc.p(float(training_phases_mat[qubit, 1]), qubit)
        qc.h(qubit)
        qc.p(float(input_mat[qubit, 2]), qubit)
        qc.p(float(training_phases_mat[qubit, 2]), qubit)
    
    return qc

def objective_function(training_phases_flat):
    qc_train = create_circuit_with_phases(input_matrix, training_phases_flat)
    state = Statevector.from_instruction(qc_train)
    probs = state.probabilities()
    target_prob = probs[target_index]
    loss = -np.log(target_prob + 1e-10)
    loss_history.append(loss)
    prob_history.append(target_prob)
    return -target_prob

initial_phases = training_matrix.flatten() * 2 * np.pi

print("Optimizing...")
result = minimize(
    objective_function,
    initial_phases,
    method='L-BFGS-B',
    bounds=[(0, 2*np.pi) for _ in range(9)],
    options={'maxiter': 100}
)

optimized_phases = result.x
final_training_matrix = optimized_phases.reshape(3, 3) / (2 * np.pi)

# 7. FINAL RESULTS
print("\n\n6. FINAL RESULTS")
print("=" * 60)

# Create final optimized circuit
qc_final = create_circuit_with_phases(input_matrix, optimized_phases)
final_state = Statevector.from_instruction(qc_final)
final_probs = final_state.probabilities()
final_target_prob = float(final_probs[target_index])

print(f"Initial probability: {target_prob_initial:.2%}")
print(f"Final probability: {final_target_prob:.2%}")
print(f"Improvement: {(final_target_prob - target_prob_initial)*100:.2f} percentage points")

print("\nOptimized Training Matrix:")
print(final_training_matrix)

# Final measurement
qc_final.measure_all()
compiled_final = transpile(qc_final, simulator)
job_final = simulator.run(compiled_final, shots=1000)
result_final = job_final.result()
counts_final = result_final.get_counts(compiled_final)

print("\nFinal measurement results:")
for state, count in sorted(counts_final.items(), key=lambda x: x[1], reverse=True):
    if state == target_state:
        print(f"**|{state}>: {count/1000:.2%} [TARGET STATE]**")
    else:
        print(f"  |{state}>: {count/1000:.2%}")

# Visualizations
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

# Loss curve
ax1.plot(loss_history)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss')
ax1.set_title('Optimization Loss')
ax1.grid(True, alpha=0.3)

# Initial distribution
states_init = list(counts.keys())
probs_init = [counts[s]/1000 for s in states_init]
bars1 = ax2.bar(states_init, probs_init, color='lightblue')
if target_state in states_init:
    bars1[states_init.index(target_state)].set_color('orange')
ax2.set_xlabel('States')
ax2.set_ylabel('Probability')
ax2.set_title('Initial Distribution')
ax2.set_xticklabels(states_init, rotation=45)

# Final distribution
states_final = list(counts_final.keys())
probs_final = [counts_final[s]/1000 for s in states_final]
bars2 = ax3.bar(states_final, probs_final, color='lightgreen')
if target_state in states_final:
    bars2[states_final.index(target_state)].set_color('darkgreen')
ax3.set_xlabel('States')
ax3.set_ylabel('Probability')
ax3.set_title('Final Distribution')
ax3.set_xticklabels(states_final, rotation=45)

plt.tight_layout()
plt.show()

print("\nOptimized Circuit:")
print(qc_final.draw(output='text'))

In [ ]:
# ENTANGLEMENT: Add 3 more qubits and entangle with trained circuit
print("\n\n7. ENTANGLEMENT WITH 3 ADDITIONAL QUBITS")
print("=" * 60)

# Create a new circuit with 6 qubits (3 original + 3 new)
entangled_circuit = QuantumCircuit(6)

# Apply the optimized circuit to first 3 qubits
for qubit in range(3):
    # IP-TP(optimized)
    entangled_circuit.p(float(input_matrix[qubit, 0]), qubit)
    entangled_circuit.p(float(final_training_matrix[qubit, 0] * 2 * np.pi), qubit)
    # H
    entangled_circuit.h(qubit)
    # IP-TP(optimized)
    entangled_circuit.p(float(input_matrix[qubit, 1]), qubit)
    entangled_circuit.p(float(final_training_matrix[qubit, 1] * 2 * np.pi), qubit)
    # H
    entangled_circuit.h(qubit)
    # IP-TP(optimized)
    entangled_circuit.p(float(input_matrix[qubit, 2]), qubit)
    entangled_circuit.p(float(final_training_matrix[qubit, 2] * 2 * np.pi), qubit)

# Add entanglement: Each of the first 3 qubits entangles with corresponding next 3 qubits
# q0 -> q3, q1 -> q4, q2 -> q5
for i in range(3):
    entangled_circuit.cx(i, i + 3)  # CNOT gate from qubit i to qubit i+3

print("Entangled circuit (first 3 qubits trained, entangled with qubits 3-5):")
print("Entanglement pattern: q0->q3, q1->q4, q2->q5")
print("\nCircuit visualization:")

# Get the circuit drawing and print line by line to avoid truncation
circuit_drawing = entangled_circuit.draw(output='text', fold=-1)
print(circuit_drawing)

# Show the statevector of the entangled system
entangled_state = Statevector.from_instruction(entangled_circuit)
entangled_probs = entangled_state.probabilities()

print(f"\nEntangled system statistics:")
print(f"- Total qubits: {entangled_circuit.num_qubits}")
print(f"- Total possible states: {2**entangled_circuit.num_qubits}")
print(f"- Non-zero probability states: {np.count_nonzero(entangled_probs > 1e-10)}")

# Show top 10 most probable states (more details)
state_indices = np.argsort(entangled_probs)[::-1][:10]
print("\nTop 10 most probable entangled states:")
print("Rank | State      | Probability | Percentage")
print("-" * 45)
for i, state_idx in enumerate(state_indices):
    state_binary = format(state_idx, '06b')
    prob = entangled_probs[state_idx]
    print(f"{i+1:2d}   | |{state_binary}> | {prob:.6f}  | {prob*100:.3f}%")

# Also show all non-zero probability states
non_zero_indices = np.where(entangled_probs > 1e-10)[0]
print(f"\nAll {len(non_zero_indices)} non-zero probability states:")
print("State      | Probability")
print("-" * 25)
for state_idx in non_zero_indices:
    state_binary = format(state_idx, '06b')
    prob = entangled_probs[state_idx]
    print(f"|{state_binary}> | {prob:.6f}")

In [ ]:
# Single target qubit entangled with all 3 source qubits
print("\n\n8. SINGLE TARGET QUBIT ENTANGLED WITH ALL 3 SOURCE QUBITS")
print("=" * 60)

# Create circuit with 4 qubits (3 source + 1 target)
single_target_circuit = QuantumCircuit(4, 1)  # 4 qubits, 1 classical bit for measurement

# First apply the optimized circuit to first 3 qubits
for qubit in range(3):
    # IP-TP(optimized)
    single_target_circuit.p(float(input_matrix[qubit, 0]), qubit)
    single_target_circuit.p(float(final_training_matrix[qubit, 0] * 2 * np.pi), qubit)
    # H
    single_target_circuit.h(qubit)
    # IP-TP(optimized)
    single_target_circuit.p(float(input_matrix[qubit, 1]), qubit)
    single_target_circuit.p(float(final_training_matrix[qubit, 1] * 2 * np.pi), qubit)
    # H
    single_target_circuit.h(qubit)
    # IP-TP(optimized)
    single_target_circuit.p(float(input_matrix[qubit, 2]), qubit)
    single_target_circuit.p(float(final_training_matrix[qubit, 2] * 2 * np.pi), qubit)

# Entangle ALL 3 source qubits with the single target qubit (qubit 3)
print("Entanglement pattern: q0->q3, q1->q3, q2->q3")
single_target_circuit.cx(0, 3)  # q0 controls q3
single_target_circuit.cx(1, 3)  # q1 controls q3
single_target_circuit.cx(2, 3)  # q2 controls q3

# Add P-H-P-H-P pattern to target qubit with trainable phases
target_phases = np.random.rand(3) * 2 * np.pi
print(f"\nTarget qubit training phases: {target_phases / (2 * np.pi)}")

# P-H-P-H-P on qubit 3
single_target_circuit.p(target_phases[0], 3)
single_target_circuit.h(3)
single_target_circuit.p(target_phases[1], 3)
single_target_circuit.h(3)
single_target_circuit.p(target_phases[2], 3)

# Measure only the target qubit
single_target_circuit.measure(3, 0)

print("\nCircuit visualization:")
print(single_target_circuit.draw(output='text', fold=-1))

# Simulate the circuit
simulator = AerSimulator()
compiled = transpile(single_target_circuit, simulator)
job = simulator.run(compiled, shots=1000)
result = job.result()
counts = result.get_counts(compiled)

print("\nMeasurement results for target qubit:")
print("-" * 30)
for outcome, count in sorted(counts.items()):
    probability = count / 1000
    print(f"Target qubit = |{outcome}>: {count} counts ({probability:.2%})")

# Analyze the state before measurement
# Create same circuit without measurement for state analysis
analysis_circuit = QuantumCircuit(4)
# Apply same operations
for qubit in range(3):
    analysis_circuit.p(float(input_matrix[qubit, 0]), qubit)
    analysis_circuit.p(float(final_training_matrix[qubit, 0] * 2 * np.pi), qubit)
    analysis_circuit.h(qubit)
    analysis_circuit.p(float(input_matrix[qubit, 1]), qubit)
    analysis_circuit.p(float(final_training_matrix[qubit, 1] * 2 * np.pi), qubit)
    analysis_circuit.h(qubit)
    analysis_circuit.p(float(input_matrix[qubit, 2]), qubit)
    analysis_circuit.p(float(final_training_matrix[qubit, 2] * 2 * np.pi), qubit)

analysis_circuit.cx(0, 3)
analysis_circuit.cx(1, 3)
analysis_circuit.cx(2, 3)

analysis_circuit.p(target_phases[0], 3)
analysis_circuit.h(3)
analysis_circuit.p(target_phases[1], 3)
analysis_circuit.h(3)
analysis_circuit.p(target_phases[2], 3)

# Get full state
full_state = Statevector.from_instruction(analysis_circuit)
full_probs = full_state.probabilities()

print("\nCorrelations between source and target:")
print("Source (q0q1q2) | Target (q3) | Probability")
print("-" * 45)
for i in range(16):  # 2^4 = 16 states
    if full_probs[i] > 1e-10:
        state_binary = format(i, '04b')
        source_state = state_binary[:3]
        target_state = state_binary[3]
        print(f"    |{source_state}>    |     |{target_state}>    | {full_probs[i]:.6f}")

In [ ]:
# Generate two different input matrices and find their most probable states
print("\n\n9. TWO INPUT MATRICES WITH DIFFERENT MOST PROBABLE STATES")
print("=" * 60)

# Generate two random input matrices and one training matrix
input_matrix_1 = np.random.rand(3, 3)
input_matrix_2 = np.random.rand(3, 3)
shared_training_matrix = np.random.rand(3, 3)

print("Input Matrix 1:")
print(input_matrix_1)
print("\nInput Matrix 2:")
print(input_matrix_2)
print("\nShared Training Matrix:")
print(shared_training_matrix)

# Function to create circuit with given input and training matrices
def create_circuit_ip_tp(input_mat, training_mat):
    qc = QuantumCircuit(3)
    for qubit in range(3):
        # IP-TP
        qc.p(float(input_mat[qubit, 0]), qubit)
        qc.p(float(training_mat[qubit, 0]), qubit)
        # H
        qc.h(qubit)
        # IP-TP
        qc.p(float(input_mat[qubit, 1]), qubit)
        qc.p(float(training_mat[qubit, 1]), qubit)
        # H
        qc.h(qubit)
        # IP-TP
        qc.p(float(input_mat[qubit, 2]), qubit)
        qc.p(float(training_mat[qubit, 2]), qubit)
    return qc

# Keep generating until we get different most probable states
max_attempts = 100
attempt = 0
most_prob_state_1 = None
most_prob_state_2 = None

while attempt < max_attempts:
    # Create circuits with both input matrices
    qc_1 = create_circuit_ip_tp(input_matrix_1, shared_training_matrix)
    qc_2 = create_circuit_ip_tp(input_matrix_2, shared_training_matrix)
    
    # Get statevectors and probabilities
    state_1 = Statevector.from_instruction(qc_1)
    probs_1 = state_1.probabilities()
    
    state_2 = Statevector.from_instruction(qc_2)
    probs_2 = state_2.probabilities()
    
    # Find most probable states
    most_prob_idx_1 = np.argmax(probs_1)
    most_prob_idx_2 = np.argmax(probs_2)
    
    most_prob_state_1 = format(most_prob_idx_1, '03b')
    most_prob_state_2 = format(most_prob_idx_2, '03b')
    
    # Check if they are different
    if most_prob_state_1 != most_prob_state_2:
        print(f"\nSuccess! Found different most probable states after {attempt + 1} attempts")
        break
    
    # If not different, regenerate matrices
    input_matrix_1 = np.random.rand(3, 3)
    input_matrix_2 = np.random.rand(3, 3)
    shared_training_matrix = np.random.rand(3, 3)
    attempt += 1

if most_prob_state_1 == most_prob_state_2:
    print(f"\nWarning: Could not find different states after {max_attempts} attempts")

# Show results
print("\n" + "="*50)
print("RESULTS WITH INPUT MATRIX 1:")
print("-"*30)

# Measure circuit 1
qc_1.measure_all()
compiled_1 = transpile(qc_1, simulator)
job_1 = simulator.run(compiled_1, shots=1000)
counts_1 = job_1.result().get_counts(compiled_1)

print("Probability distribution:")
for state, count in sorted(counts_1.items(), key=lambda x: x[1], reverse=True):
    prob = count/1000
    if state.replace(' ', '') == most_prob_state_1:
        print(f"**|{state}>: {prob:.2%} [MOST PROBABLE]**")
    else:
        print(f"  |{state}>: {prob:.2%}")

print(f"\nMost probable state for Input Matrix 1: |{most_prob_state_1}>")
print(f"Theoretical probability: {probs_1[most_prob_idx_1]:.4f}")

print("\n" + "="*50)
print("RESULTS WITH INPUT MATRIX 2:")
print("-"*30)

# Measure circuit 2
qc_2.measure_all()
compiled_2 = transpile(qc_2, simulator)
job_2 = simulator.run(compiled_2, shots=1000)
counts_2 = job_2.result().get_counts(compiled_2)

print("Probability distribution:")
for state, count in sorted(counts_2.items(), key=lambda x: x[1], reverse=True):
    prob = count/1000
    if state.replace(' ', '') == most_prob_state_2:
        print(f"**|{state}>: {prob:.2%} [MOST PROBABLE]**")
    else:
        print(f"  |{state}>: {prob:.2%}")

print(f"\nMost probable state for Input Matrix 2: |{most_prob_state_2}>")
print(f"Theoretical probability: {probs_2[most_prob_idx_2]:.4f}")

# Store the states
stored_states = {
    "input_matrix_1": {
        "matrix": input_matrix_1,
        "most_probable_state": most_prob_state_1,
        "probability": float(probs_1[most_prob_idx_1])
    },
    "input_matrix_2": {
        "matrix": input_matrix_2,
        "most_probable_state": most_prob_state_2,
        "probability": float(probs_2[most_prob_idx_2])
    },
    "shared_training_matrix": shared_training_matrix
}

print("\n" + "="*50)
print("STORED RESULTS:")
print(f"Input Matrix 1 → Most probable state: |{most_prob_state_1}> (p={stored_states['input_matrix_1']['probability']:.4f})")
print(f"Input Matrix 2 → Most probable state: |{most_prob_state_2}> (p={stored_states['input_matrix_2']['probability']:.4f})")
print(f"States are different: {most_prob_state_1 != most_prob_state_2}")

In [ ]:
# Train for Input Matrix 1's most probable state
print("\n\n10. TRAINING FOR INPUT MATRIX 1'S TARGET STATE")
print("=" * 60)

# Extract stored data
target_state_1 = stored_states["input_matrix_1"]["most_probable_state"]
target_idx_1 = int(target_state_1, 2)
input_mat_1 = stored_states["input_matrix_1"]["matrix"]
training_mat_initial = stored_states["shared_training_matrix"].copy()

print(f"Target state for Matrix 1: |{target_state_1}>")
print(f"Initial probability: {stored_states['input_matrix_1']['probability']:.4f}")

# Optimization for Matrix 1
loss_history_1 = []
prob_history_1 = []

def objective_matrix_1(training_phases_flat):
    qc = create_circuit_ip_tp(input_mat_1, training_phases_flat.reshape(3, 3))
    state = Statevector.from_instruction(qc)
    probs = state.probabilities()
    target_prob = probs[target_idx_1]
    loss = -np.log(target_prob + 1e-10)
    loss_history_1.append(loss)
    prob_history_1.append(target_prob)
    return -target_prob

# Initial phases
initial_phases_1 = training_mat_initial.flatten() * 2 * np.pi

print("\nOptimizing training matrix for Input Matrix 1...")
result_1 = minimize(
    objective_matrix_1,
    initial_phases_1,
    method='L-BFGS-B',
    bounds=[(0, 2*np.pi) for _ in range(9)],
    options={'maxiter': 100}
)

# Get optimized training matrix
optimized_phases_1 = result_1.x
trained_matrix_1 = optimized_phases_1.reshape(3, 3) / (2 * np.pi)

# Evaluate final result
qc_final_1 = create_circuit_ip_tp(input_mat_1, optimized_phases_1.reshape(3, 3))
final_state_1 = Statevector.from_instruction(qc_final_1)
final_probs_1 = final_state_1.probabilities()
final_prob_1 = float(final_probs_1[target_idx_1])

print(f"\nTraining complete for Matrix 1!")
print(f"Initial probability of |{target_state_1}>: {stored_states['input_matrix_1']['probability']:.4f}")
print(f"Final probability of |{target_state_1}>: {final_prob_1:.4f}")
print(f"Improvement: {(final_prob_1 - stored_states['input_matrix_1']['probability'])*100:.2f} percentage points")

print("\nTrained matrix after optimizing for Input Matrix 1:")
print(trained_matrix_1)

# Measure to verify
qc_final_1.measure_all()
compiled = transpile(qc_final_1, simulator)
job = simulator.run(compiled, shots=1000)
counts = job.result().get_counts(compiled)

print("\nMeasurement verification (1000 shots):")
for state, count in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:5]:
    prob = count/1000
    if state.replace(' ', '') == target_state_1:
        print(f"**|{state}>: {prob:.2%} [TARGET]**")
    else:
        print(f"  |{state}>: {prob:.2%}")

# Store the trained matrix
trained_matrix_after_1 = trained_matrix_1.copy()

In [ ]:
# Continue training for Input Matrix 2's state using the already trained matrix
print("\n\n11. CONTINUING TRAINING FOR INPUT MATRIX 2'S TARGET STATE")
print("=" * 60)

# Extract data for Matrix 2
target_state_2 = stored_states["input_matrix_2"]["most_probable_state"]
target_idx_2 = int(target_state_2, 2)
input_mat_2 = stored_states["input_matrix_2"]["matrix"]

print(f"Target state for Matrix 2: |{target_state_2}>")

# First check what happens with Matrix 2 using the trained matrix from Matrix 1
qc_check = create_circuit_ip_tp(input_mat_2, trained_matrix_after_1 * 2 * np.pi)
check_state = Statevector.from_instruction(qc_check)
check_probs = check_state.probabilities()
check_prob_2 = float(check_probs[target_idx_2])

print(f"\nUsing Matrix 1's trained parameters with Input Matrix 2:")
print(f"Probability of Matrix 2's target |{target_state_2}>: {check_prob_2:.4f}")
print(f"Original probability was: {stored_states['input_matrix_2']['probability']:.4f}")

# Now continue training for Matrix 2's target
loss_history_2 = []
prob_history_2 = []

def objective_matrix_2(training_phases_flat):
    qc = create_circuit_ip_tp(input_mat_2, training_phases_flat.reshape(3, 3))
    state = Statevector.from_instruction(qc)
    probs = state.probabilities()
    target_prob = probs[target_idx_2]
    loss = -np.log(target_prob + 1e-10)
    loss_history_2.append(loss)
    prob_history_2.append(target_prob)
    return -target_prob

# Start from the trained matrix from Matrix 1
initial_phases_2 = trained_matrix_after_1.flatten() * 2 * np.pi

print("\nContinuing optimization for Input Matrix 2...")
print("Starting from Matrix 1's trained parameters...")

result_2 = minimize(
    objective_matrix_2,
    initial_phases_2,
    method='L-BFGS-B',
    bounds=[(0, 2*np.pi) for _ in range(9)],
    options={'maxiter': 100}
)

# Get final optimized training matrix
optimized_phases_2 = result_2.x
trained_matrix_2 = optimized_phases_2.reshape(3, 3) / (2 * np.pi)

# Evaluate final result for Matrix 2
qc_final_2 = create_circuit_ip_tp(input_mat_2, optimized_phases_2.reshape(3, 3))
final_state_2 = Statevector.from_instruction(qc_final_2)
final_probs_2 = final_state_2.probabilities()
final_prob_2 = float(final_probs_2[target_idx_2])

print(f"\nTraining complete for Matrix 2!")
print(f"Starting probability (with Matrix 1's training): {check_prob_2:.4f}")
print(f"Final probability of |{target_state_2}>: {final_prob_2:.4f}")
print(f"Total improvement: {(final_prob_2 - stored_states['input_matrix_2']['probability'])*100:.2f} percentage points")

# Check what happened to Matrix 1's performance
qc_check_1 = create_circuit_ip_tp(input_mat_1, optimized_phases_2.reshape(3, 3))
check_state_1 = Statevector.from_instruction(qc_check_1)
check_probs_1 = check_state_1.probabilities()
final_prob_1_after_2 = float(check_probs_1[target_idx_1])

print(f"\nEffect on Matrix 1's target state |{target_state_1}>:")
print(f"After training for Matrix 1 only: {final_prob_1:.4f}")
print(f"After additional training for Matrix 2: {final_prob_1_after_2:.4f}")
print(f"Change: {(final_prob_1_after_2 - final_prob_1)*100:+.2f} percentage points")

print("\nFinal trained matrix (after both trainings):")
print(trained_matrix_2)

# Verify with measurements
print("\n" + "="*50)
print("VERIFICATION MEASUREMENTS:")

# Measure Matrix 2 with final parameters
qc_final_2.measure_all()
compiled_2 = transpile(qc_final_2, simulator)
job_2 = simulator.run(compiled_2, shots=1000)
counts_2 = job_2.result().get_counts(compiled_2)

print("\nMatrix 2 with final parameters (1000 shots):")
for state, count in sorted(counts_2.items(), key=lambda x: x[1], reverse=True)[:5]:
    prob = count/1000
    if state.replace(' ', '') == target_state_2:
        print(f"**|{state}>: {prob:.2%} [TARGET]**")
    else:
        print(f"  |{state}>: {prob:.2%}")

# Measure Matrix 1 with final parameters
qc_check_1.measure_all()
compiled_1 = transpile(qc_check_1, simulator)
job_1 = simulator.run(compiled_1, shots=1000)
counts_1 = job_1.result().get_counts(compiled_1)

print("\nMatrix 1 with final parameters (1000 shots):")
for state, count in sorted(counts_1.items(), key=lambda x: x[1], reverse=True)[:5]:
    prob = count/1000
    if state.replace(' ', '') == target_state_1:
        print(f"**|{state}>: {prob:.2%} [ORIGINAL TARGET]**")
    else:
        print(f"  |{state}>: {prob:.2%}")

In [ ]:
# Final comparison: Both input matrices with the final trained matrix
print("\n\n12. FINAL PROBABILITY DISTRIBUTIONS WITH FINAL TRAINED MATRIX")
print("=" * 60)

print("Final trained matrix (after training for both targets):")
print(trained_matrix_2)
print("\n" + "="*60)

# Create circuits with final trained matrix
final_trained_phases = trained_matrix_2 * 2 * np.pi

qc_matrix1_final = create_circuit_ip_tp(input_mat_1, final_trained_phases)
qc_matrix2_final = create_circuit_ip_tp(input_mat_2, final_trained_phases)

# Get statevectors for analysis
state_1_final = Statevector.from_instruction(qc_matrix1_final)
state_2_final = Statevector.from_instruction(qc_matrix2_final)

probs_1_final = state_1_final.probabilities()
probs_2_final = state_2_final.probabilities()

# Measure both circuits
qc_matrix1_final.measure_all()
qc_matrix2_final.measure_all()

compiled_1_final = transpile(qc_matrix1_final, simulator)
compiled_2_final = transpile(qc_matrix2_final, simulator)

job_1_final = simulator.run(compiled_1_final, shots=1000)
job_2_final = simulator.run(compiled_2_final, shots=1000)

counts_1_final = job_1_final.result().get_counts(compiled_1_final)
counts_2_final = job_2_final.result().get_counts(compiled_2_final)

# Display results side by side
print("\nINPUT MATRIX 1 - PROBABILITY DISTRIBUTION")
print("-" * 50)
print(f"Target state: |{target_state_1}>")
print(f"Theoretical probability: {probs_1_final[target_idx_1]:.4f}")
print("\nMeasurement results (1000 shots):")
print("State | Count | Probability")
print("-" * 30)
for state, count in sorted(counts_1_final.items(), key=lambda x: x[1], reverse=True):
    prob = count/1000
    state_clean = state.replace(' ', '')
    if state_clean == target_state_1:
        print(f"|{state}> | {count:4d} | {prob:6.2%} **[ORIGINAL TARGET]**")
    else:
        print(f"|{state}> | {count:4d} | {prob:6.2%}")

print("\n" + "="*60)

print("\nINPUT MATRIX 2 - PROBABILITY DISTRIBUTION")
print("-" * 50)
print(f"Target state: |{target_state_2}>")
print(f"Theoretical probability: {probs_2_final[target_idx_2]:.4f}")
print("\nMeasurement results (1000 shots):")
print("State | Count | Probability")
print("-" * 30)
for state, count in sorted(counts_2_final.items(), key=lambda x: x[1], reverse=True):
    prob = count/1000
    state_clean = state.replace(' ', '')
    if state_clean == target_state_2:
        print(f"|{state}> | {count:4d} | {prob:6.2%} **[CURRENT TARGET]**")
    else:
        print(f"|{state}> | {count:4d} | {prob:6.2%}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Matrix 1 distribution
states_1 = list(counts_1_final.keys())
probs_1 = [counts_1_final[s]/1000 for s in states_1]
bars1 = ax1.bar(range(len(states_1)), probs_1, color='lightblue')

# Highlight target state
for i, state in enumerate(states_1):
    if state.replace(' ', '') == target_state_1:
        bars1[i].set_color('darkblue')
        bars1[i].set_edgecolor('black')
        bars1[i].set_linewidth(2)

ax1.set_xticks(range(len(states_1)))
ax1.set_xticklabels([s.replace(' ', '') for s in states_1], rotation=45)
ax1.set_xlabel('Quantum States')
ax1.set_ylabel('Probability')
ax1.set_title(f'Input Matrix 1\\nTarget: |{target_state_1}>')
ax1.grid(axis='y', alpha=0.3)

# Add percentage labels
for i, prob in enumerate(probs_1):
    ax1.text(i, prob + 0.01, f'{prob:.1%}', ha='center', va='bottom', fontsize=9)

# Matrix 2 distribution
states_2 = list(counts_2_final.keys())
probs_2 = [counts_2_final[s]/1000 for s in states_2]
bars2 = ax2.bar(range(len(states_2)), probs_2, color='lightgreen')

# Highlight target state
for i, state in enumerate(states_2):
    if state.replace(' ', '') == target_state_2:
        bars2[i].set_color('darkgreen')
        bars2[i].set_edgecolor('black')
        bars2[i].set_linewidth(2)

ax2.set_xticks(range(len(states_2)))
ax2.set_xticklabels([s.replace(' ', '') for s in states_2], rotation=45)
ax2.set_xlabel('Quantum States')
ax2.set_ylabel('Probability')
ax2.set_title(f'Input Matrix 2\\nTarget: |{target_state_2}>')
ax2.grid(axis='y', alpha=0.3)

# Add percentage labels
for i, prob in enumerate(probs_2):
    ax2.text(i, prob + 0.01, f'{prob:.1%}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Final Probability Distributions with Shared Trained Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Summary
print("\n" + "="*60)
print("SUMMARY:")
print(f"Matrix 1 target |{target_state_1}> probability: {probs_1_final[target_idx_1]:.4f} ({counts_1_final.get(target_state_1.replace('', ' ').strip(), 0)/10:.1%} measured)")
print(f"Matrix 2 target |{target_state_2}> probability: {probs_2_final[target_idx_2]:.4f} ({counts_2_final.get(target_state_2.replace('', ' ').strip(), 0)/10:.1%} measured)")
print("\nThe final trained matrix balances between both targets, with preference for Matrix 2's target")
print("since it was trained last.")